# ASVspoof 5 Track 1 - Colab 학습 노트북 (50% train subset)

`ASVspoof5_colab.ipynb`에서 다음을 바꾼 버전이다.
- 증강 추가: codec(mp3 / aac / amr-nb / narrowband-8k), MUSAN, RIR (eval 코덱 조건 C01–C11 대응)
- best ckpt 선택 기준을 dev minDCF로 변경, Cllr도 같이 기록
- train을 (key, attack_tag) 기준 층화 subset으로 줄임 (기본 50%, 약 91K 발화). dev/eval은 그대로
- W&B 기록, eval 다운로드/평가, 공격·코덱별 분해, actDCF 추가

subset을 쓰는 이유: full train은 T4에서 epoch당 약 3.5시간이라 12시간 세션에 2–3 epoch밖에 못 돈다. 동결 백본 + 작은 head 구조라 절반 데이터로도 수렴하고(ASVspoof 2019 LA 전체의 약 3.6배), 같은 시간에 훨씬 많은 epoch을 돌릴 수 있다.

실험 조합(COMBO), epoch, subset 비율은 셀 9.5에서 한 번에 정한다. 저장 경로는 `ckpt/<COMBO>_v2_subset50_epoch<N>/` 형식.

사전 준비
- 코드는 Drive `MyDrive/STUDY/Project/ASVspoof5/`에 둔다.
- 데이터셋이 gated일 수 있으므로 `huggingface-cli login` 또는 `HF_TOKEN` 설정.
- GPU 런타임 필요. 추출된 데이터가 약 83 GB라 디스크가 큰 런타임을 쓴다.

## 셀 2 - GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Runtime > Change runtime type > GPU 선택 후 다시 실행하세요.")
p = torch.cuda.get_device_properties(0)
print(f"GPU       : {p.name}")
print(f"VRAM      : {p.total_memory / 1024**3:.1f} GB")
print(f"PyTorch   : {torch.__version__}")
if p.total_memory / 1024**3 < 14.0:
    print("WavLM-Large frozen+batch 8은 약 11GB 사용. 15GB 미만 GPU는 batch 4로 자동 축소됨.")

## 셀 3 - Google Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 셀 4 - 경로 / 환경 변수
`DRIVE_ROOT`만 본인 Drive 경로로 바꾸면 된다.

In [ ]:
import os

# 사용자 설정
GITHUB_REPO = "https://github.com/Eunchae-L/ASVspoof.git"   # (참고용) 코드는 Drive에 영구 보관됨
DRIVE_ROOT  = "/content/drive/MyDrive/STUDY/Project/ASVspoof5"

PROJECT_ROOT = DRIVE_ROOT                     # 코드 위치 (Drive, 영구 - 모든 .py/configs/scripts 여기 있음)
DATA_ROOT    = "/content/data/asvspoof5"      # 추출된 flac/protocol (세션 종료 시 사라짐)
RAW_DIR      = "/content/raw"                 # 다운받은 tar (휘발! ~54GB - Drive 절약, 세션마다 재다운로드)
HF_HOME      = f"{DRIVE_ROOT}/.hf_cache"      # WavLM 모델 캐시 (영구, ~1.3 GB)
CKPT_DIR     = "/content/ckpt"                # 학습 중 ckpt (휘발, 빠른 디스크)
CKPT_BACKUP  = f"{DRIVE_ROOT}/ckpt"           # ckpt 백업 (영구)

os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HOME

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(CKPT_BACKUP, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}   (=DRIVE_ROOT, 코드 위치)")
print(f"RAW_DIR      = {RAW_DIR}   (휘발 - Drive 절약, 세션마다 재다운로드)")
print(f"DATA_ROOT    = {DATA_ROOT}  (세션마다 추출)")
print(f"HF_HOME      = {HF_HOME}")
print(f"CKPT_DIR     = {CKPT_DIR}   (휘발)")
print(f"CKPT_BACKUP  = {CKPT_BACKUP}   (영구)")

## 셀 5 - 작업 디렉터리 이동
코드는 Drive에 있으므로 cd만 한다.

In [ ]:
import os

# 코드는 Drive에 영구 보관됨. cwd만 PROJECT_ROOT로 이동.
os.chdir(PROJECT_ROOT)
print(f"cwd = {os.getcwd()}")
!ls -la

# (선택) GitHub에서 최신 변경사항을 Drive로 pull 받고 싶다면 아래 주석 해제:
# if os.path.isdir(f"{PROJECT_ROOT}/.git"):
#     !cd $PROJECT_ROOT && git pull --ff-only

## 셀 6 - 의존성 설치

In [ ]:
!pip install -q "transformers>=4.40" "huggingface_hub>=0.23" librosa soundfile pyyaml pandas

import transformers, huggingface_hub, librosa, soundfile, yaml, torch, torchaudio, sklearn, scipy, numpy, pandas
print(f"transformers     {transformers.__version__}")
print(f"huggingface_hub  {huggingface_hub.__version__}")
print(f"librosa          {librosa.__version__}")
print(f"soundfile        {soundfile.__version__}")
print(f"pandas           {pandas.__version__}")
print(f"pyyaml           {yaml.__version__}")
print(f"torch            {torch.__version__}")
print(f"torchaudio       {torchaudio.__version__}")
print(f"scikit-learn     {sklearn.__version__}")
print(f"scipy            {scipy.__version__}")
print(f"numpy            {numpy.__version__}")

## 셀 6.5 - W&B 설정
loss / lr / grad_norm, epoch별 EER·minDCF·Cllr를 W&B에 기록한다.
API key는 Colab Secrets에 `WANDB_API_KEY`로 등록해 두면 자동으로 읽는다. 로그인이 안 되면 offline으로 돌고, 어느 경우든 `train_log.jsonl`에는 로컬로 남는다.
W&B를 안 쓸 거면 이 셀을 건너뛰고 셀 13에서 `--wandb` 대신 `--no-wandb`를 쓴다.

In [ ]:
!pip install -q "wandb>=0.16"
import wandb, os

# W&B 프로젝트 이름 (대시보드에서 이 이름으로 묶여 보임)
WANDB_PROJECT = "asvspoof5-track1"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

# 로그인 시도. key가 없거나 로그인 실패해도 학습은 진행됨(로컬 JSONL fallback).
#   - Colab Secrets에 WANDB_API_KEY를 등록해두면 자동 인식
#   - 아니면 아래 실행 후 출력되는 링크에서 key를 복사해 붙여넣기
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        wandb.login(key=key)
    else:
        wandb.login()
except Exception:
    try:
        wandb.login()
    except Exception as e:
        print(f"[wandb] 로그인 건너뜀 ({e}). 오프라인 모드로 진행 → 셀 13에서 로컬 로그만 남습니다.")
        os.environ["WANDB_MODE"] = "offline"

print(f"[wandb] project={WANDB_PROJECT}  mode={os.environ.get('WANDB_MODE', 'online')}")
print("[wandb] 로그인 실패/미설정이어도 train.py는 자동으로 로컬 train_log.jsonl에 기록합니다.")

## 셀 7 - 데이터 다운로드 + 추출 (매 세션, 30–60분)
shard를 하나씩 받아서 바로 풀고 tar를 지운다. 디스크 최대 사용량이 "추출된 flac(약 83 GB) + tar 1개"로 유지된다.
추출 결과는 `/content`에만 두므로 standard 런타임(약 78 GB)에서는 부족할 수 있다.

In [ ]:
import os, subprocess, shutil
from huggingface_hub import HfApi, hf_hub_download

REPO = "jungjee/asvspoof5"

def disk_free_gb(p="/content"):
    t, u, f = shutil.disk_usage(p); return f / 1024**3

def find_dir(name, root):
    for r, dirs, _ in os.walk(root):
        if name in dirs:
            return os.path.join(r, name)
    return None

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

if find_dir("flac_T", DATA_ROOT) and find_dir("flac_D", DATA_ROOT):
    print("[data] flac 이미 추출됨 → skip")
else:
    api = HfApi()
    files = api.list_repo_files(REPO, repo_type="dataset")
    protocols = sorted(f for f in files if f.startswith("ASVspoof5_protocols.tar"))
    train_sh  = sorted(f for f in files if f.startswith("flac_T_") and f.endswith(".tar"))
    dev_sh    = sorted(f for f in files if f.startswith("flac_D_") and f.endswith(".tar"))
    print(f"[stream] protocols={len(protocols)}  flac_T={len(train_sh)}  flac_D={len(dev_sh)}")

    def stream(fname):
        print(f"  {fname:<28} (free {disk_free_gb():5.0f}GB)", flush=True)
        path = hf_hub_download(REPO, fname, repo_type="dataset", local_dir=RAW_DIR)
        flag = "-xzf" if path.endswith(".gz") else "-xf"
        subprocess.run(["tar", flag, path, "-C", DATA_ROOT], check=True)
        os.remove(path)                          # 추출 즉시 tar 삭제 -> 디스크 절약
        shutil.rmtree(os.path.join(RAW_DIR, ".cache"), ignore_errors=True)  # hf 임시 blob 정리
        print(f"    extracted + tar 삭제, free {disk_free_gb():.0f}GB", flush=True)

    stream(protocols[0])                          # .tar.gz 우선 (정렬상 .gz가 뒤이면 조정)
    for f in train_sh: stream(f)
    for f in dev_sh:   stream(f)
    print("[stream] done")

## 셀 8 - 추출 결과 확인
protocol tsv 위치와 행 수만 확인한다.

In [ ]:
import os, subprocess, shutil

def disk_free_gb(p="/content"):
    t, u, f = shutil.disk_usage(p); return u/1024**3, t/1024**3, f/1024**3

def find_file(name, root):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

train_tsv = find_file("ASVspoof5.train.tsv", DATA_ROOT)
dev_tsv   = find_file("ASVspoof5.dev.track_1.tsv", DATA_ROOT)

used, total, free = disk_free_gb("/content")
print(f"[disk] /content : {used:.1f}/{total:.1f} GB used, {free:.1f} GB free")
print(f"[check] train_tsv = {train_tsv}")
print(f"[check] dev_tsv   = {dev_tsv}")
if not (train_tsv and dev_tsv):
    raise FileNotFoundError("protocol tsv 못 찾음 - 셀 7(스트리밍 다운로드+추출)을 먼저 실행하세요.")

def wc(p):
    return int(subprocess.check_output(["wc", "-l", p]).split()[0])

n_train, n_dev = wc(train_tsv), wc(dev_tsv)
print(f"\ntrain protocol rows: {n_train:,}  ({train_tsv})")
print(f"dev   protocol rows: {n_dev:,}  ({dev_tsv})")
assert n_train > 180_000, "train protocol row count too low"
assert n_dev   > 140_000, "dev protocol row count too low"

expected_root = os.path.dirname(os.path.dirname(train_tsv))
if expected_root != DATA_ROOT.rstrip("/"):
    print(f"\n실제 데이터 루트는 {expected_root} (DATA_ROOT와 다름).")
    print(f"    cfg['data']['root'] = '{expected_root}' 로 셀 10에서 덮어쓰세요.")

## 셀 8.5 - MUSAN / RIRS_NOISES 다운로드, ffmpeg 확인
처음 한 번만 받아서 Drive에 둔다 (MUSAN 약 11 GB, RIRS_NOISES 약 2 GB). codec 증강에 ffmpeg가 필요하다.

In [ ]:
import os, subprocess

MUSAN_DIR = f"{DRIVE_ROOT}/musan"
RIRS_DIR  = f"{DRIVE_ROOT}/RIRS_NOISES"

# MUSAN
if os.path.isdir(f"{MUSAN_DIR}/noise"):
    print(f"[musan] 이미 있음 → skip ({MUSAN_DIR})")
else:
    print(f"[musan] downloading to {MUSAN_DIR} (~11 GB) …")
    subprocess.run(["wget", "-q", "-O", "/tmp/musan.tar.gz",
                    "http://www.openslr.org/resources/17/musan.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/musan.tar.gz", "-C", DRIVE_ROOT], check=True)
    subprocess.run(["rm", "-f", "/tmp/musan.tar.gz"], check=False)
    print(f"[musan] done")

# RIRS_NOISES
if os.path.isdir(f"{RIRS_DIR}/simulated_rirs"):
    print(f"[rirs] 이미 있음 → skip ({RIRS_DIR})")
else:
    print(f"[rirs] downloading to {RIRS_DIR} (~2 GB) …")
    subprocess.run(["wget", "-q", "-O", "/tmp/rirs.zip",
                    "http://www.openslr.org/resources/28/rirs_noises.zip"], check=True)
    subprocess.run(["unzip", "-q", "/tmp/rirs.zip", "-d", DRIVE_ROOT], check=True)
    subprocess.run(["rm", "-f", "/tmp/rirs.zip"], check=False)
    print(f"[rirs] done")

# ffmpeg sanity (필요: codec augmentation)
r = subprocess.run(["ffmpeg", "-hide_banner", "-version"],
                   capture_output=True, text=True)
print(r.stdout.splitlines()[0] if r.returncode == 0 else "[ffmpeg] NOT FOUND (codec augment will be skipped)")

print(f"\nMUSAN_DIR = {MUSAN_DIR}")
print(f"RIRS_DIR  = {RIRS_DIR}")

## 셀 9 - ckpt 디렉터리
`cfg["data"]["root"]`에 절대경로를 넣으므로 symlink는 필요 없다.

In [ ]:
import os

# cfg["data"]["root"]에 절대경로(DATA_ROOT)를 직접 넣으므로 symlink 불필요
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(CKPT_BACKUP, exist_ok=True)
print(f"CKPT_DIR    = {CKPT_DIR}")
print(f"CKPT_BACKUP = {CKPT_BACKUP}")

## 셀 9.5 - 실험 조합 선택
`COMBO`, `EPOCHS`, `SUBSET_RATIO`만 바꾸면 config 파일, TAG, 저장 경로, W&B run 이름이 모두 조합별로 나뉜다. 아래 셀들은 전부 이 값을 쓴다.

| COMBO | adapter | classifier |
|---|---|---|
| a1c1 | bilstm | lcnn_big |
| a1c2 | bilstm | senet12 |
| a2c1 | se_res2net | lcnn_big |
| a2c2 | se_res2net | senet12 |

조합을 바꿀 때는 이 셀부터 다시 실행한다. 세션 재시작 후 평가/시각화만 할 때도 이 셀을 먼저 돌려야 `TAG`가 정의된다.

In [ ]:
# 실험 조합 선택 (single source of truth)
COMBOS = {
    "a1c1": {"adapter": "bilstm",     "classifier": "lcnn_big"},
    "a1c2": {"adapter": "bilstm",     "classifier": "senet12"},
    "a2c1": {"adapter": "se_res2net", "classifier": "lcnn_big"},
    "a2c2": {"adapter": "se_res2net", "classifier": "senet12"},
}

# 실험할 때 여기 3개만 바꾸면 됨
COMBO        = "a1c1"      # a1c1 | a1c2 | a2c1 | a2c2
EPOCHS       = 20
SUBSET_RATIO = 0.5         # train subset 비율 (0.3 / 0.4 / 0.5)

assert COMBO in COMBOS, f"COMBO는 {list(COMBOS)} 중 하나"
ADAPTER    = COMBOS[COMBO]["adapter"]
CLASSIFIER = COMBOS[COMBO]["classifier"]
SUBSET_TAG = f"subset{int(SUBSET_RATIO*100)}"
TAG        = f"{COMBO}_v2_{SUBSET_TAG}_epoch{EPOCHS}"    # 저장/체크포인트/W&B run 이름
CFG_NAME   = f"colab_{COMBO}_v2_{SUBSET_TAG}.yaml"       # 조합별 전용 config (덮어쓰기 충돌 방지)
CFG        = f"{PROJECT_ROOT}/configs/{CFG_NAME}"        # 절대경로 (open/evaluate --config용)
CFG_REL    = f"configs/{CFG_NAME}"                       # 상대경로 (!python --config용, cwd=PROJECT_ROOT)

print(f"[exp] COMBO={COMBO}  adapter={ADAPTER}  classifier={CLASSIFIER}  epochs={EPOCHS}")
print(f"[exp] TAG = {TAG}")
print(f"[exp] CFG = {CFG_REL}")
print(f"[exp] ckpt→ {CKPT_DIR}/{TAG}   (Drive 백업 {CKPT_BACKUP}/{TAG})")

## 셀 10 - Colab용 config
`configs/baseline.yaml`을 읽어 Colab 환경, 증강(codec/MUSAN/RIR), 조합을 반영하고 `configs/colab_<COMBO>_v2_<SUBSET_TAG>.yaml`로 저장한다. VRAM에 따라 batch size, compute capability에 따라 AMP dtype이 정해진다.

In [ ]:
import yaml, torch, os
# '실험 선택' 셀(COMBO/ADAPTER/CLASSIFIER/EPOCHS/SUBSET_TAG/CFG)을 먼저 실행하세요.

with open(f"{PROJECT_ROOT}/configs/baseline.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"]               = DATA_ROOT
cfg["data"]["num_workers"]        = 8
cfg["data"]["pin_memory"]         = True
cfg["data"]["persistent_workers"] = True
cfg["data"]["prefetch_factor"]    = 4
cfg["train"]["save_dir"]          = CKPT_DIR
cfg["train"]["epochs"]            = EPOCHS       # ← 실험 선택 셀

def _find_tsv(name):
    for r, _, files in os.walk(DATA_ROOT):
        if name in files:
            return os.path.relpath(os.path.join(r, name), DATA_ROOT)
    raise FileNotFoundError(f"{name} not under {DATA_ROOT}")

cfg["data"]["protocol_train"] = _find_tsv("ASVspoof5.train.tsv")
cfg["data"]["protocol_dev"]   = _find_tsv("ASVspoof5.dev.track_1.tsv")
try:
    cfg["data"]["protocol_eval"] = _find_tsv("ASVspoof5.eval.track_1.tsv")
except FileNotFoundError:
    pass
# subset protocol이 이미 있으면 그걸 사용 (조합만 바꿔서 재실험할 땐 subset 재생성 불필요)
_subset_rel = f"protocols/ASVspoof5.train.{SUBSET_TAG}.tsv"
if os.path.isfile(os.path.join(DATA_ROOT, _subset_rel)):
    cfg["data"]["protocol_train"] = _subset_rel
    print(f"[subset] 기존 subset 사용 → {_subset_rel}")
print(f"protocol_train = {cfg['data']['protocol_train']}")
print(f"protocol_dev   = {cfg['data']['protocol_dev']}")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb < 14:   cfg["train"]["batch_size"] = 4
elif vram_gb < 24: cfg["train"]["batch_size"] = 8      # T4-class
elif vram_gb < 50: cfg["train"]["batch_size"] = 24     # A100 40GB
else:              cfg["train"]["batch_size"] = 48     # A100 80GB / H100
cap = torch.cuda.get_device_capability(0)
amp_dtype = "bf16" if cap[0] >= 8 else "fp16"
print(f"VRAM {vram_gb:.1f}GB ({torch.cuda.get_device_name(0)}, cap {cap[0]}.{cap[1]}) "
      f"→ batch_size {cfg['train']['batch_size']}, AMP {amp_dtype}, epochs {cfg['train']['epochs']}")

# v2 augmentation (codec + MUSAN + RIR)
cfg["augment"]["musan_root"]    = MUSAN_DIR
cfg["augment"]["musan_p"]       = 0.3
cfg["augment"]["rir_root"]      = f"{RIRS_DIR}/simulated_rirs"
cfg["augment"]["rir_p"]         = 0.2
cfg["augment"]["codec_p"]       = 0.3
cfg["augment"]["codec_choices"] = ["mp3", "aac", "amr-nb", "narrowband-8k"]

# 조합 반영 (실험 선택 셀에서)
cfg["model"]["adapter"]["type"]    = ADAPTER
cfg["model"]["classifier"]["type"] = CLASSIFIER

with open(CFG, "w") as f:                       # CFG = 조합별 전용 config 파일
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f"\nwrote {CFG}")
print(f"[arch] COMBO={COMBO}  adapter={ADAPTER}  classifier={CLASSIFIER}")
for k in ["batch_size", "epochs", "lr_peak", "early_stopping_patience"]:
    print(f"train.{k:<24}= {cfg['train'][k]}")
print(f"loss.lambda_workers      = {cfg['loss']['lambda_workers']}")

## 셀 10.5 - train subset 생성
full train protocol에서 (key, attack_tag)별로 `SUBSET_RATIO`만큼 샘플링해 bonafide/spoof 비율과 공격(A01–A08) 분포를 유지한다. 항상 full protocol에서 뽑기 때문에 subset의 subset이 생기지 않는다. dev/eval은 건드리지 않는다.

In [ ]:
import pandas as pd, yaml, os
# SUBSET_RATIO / SUBSET_TAG 는 '실험 선택' 셀에서 정의됨. subset은 조합과 무관 -> 한 번만 만들면 재사용.
SEED = 42

with open(CFG) as f:
    cfg_now = yaml.safe_load(f)
root = cfg_now["data"]["root"]

def _find(name, base):
    for r, _, fs in os.walk(base):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

# 항상 FULL train protocol에서 subset 생성 (config가 이미 subset을 가리켜도 subset의 subset 방지)
full_train_tsv = _find("ASVspoof5.train.tsv", root)

COLS = ["speaker_id","file_id","gender","codec","codec_q","codec_seed",
        "attack_tag","attack_label","key","tmp"]
df = pd.read_csv(full_train_tsv, sep=r"\s+", header=None, names=COLS, engine="python")
print(f"[full] {len(df):,} utt  "
      f"({(df['key']=='bonafide').sum():,} bona / {(df['key']=='spoof').sum():,} spoof)")

# Stratified by (key, attack_tag): 클래스 비율 + 공격 분포(A01–A08) 보존
sub = (df.groupby(["key", "attack_tag"], dropna=False, group_keys=False)
         .apply(lambda g: g.sample(frac=SUBSET_RATIO, random_state=SEED))).sort_index()
print(f"[subset {SUBSET_TAG}] {len(sub):,} utt  "
      f"({(sub['key']=='bonafide').sum():,} bona / {(sub['key']=='spoof').sum():,} spoof)")
print("[per-attack full → subset]")
for k, grp in df.groupby("attack_tag"):
    sub_n = (sub["attack_tag"] == k).sum()
    print(f"  {k:>10s}  {len(grp):>7,} → {sub_n:>6,}  ({sub_n/len(grp):.1%})")

out_rel = f"protocols/ASVspoof5.train.{SUBSET_TAG}.tsv"
out_abs = os.path.join(root, out_rel)
os.makedirs(os.path.dirname(out_abs), exist_ok=True)
sub.to_csv(out_abs, sep=" ", header=False, index=False)

# 현재 조합 config의 protocol_train 을 subset 으로 교체
cfg_now["data"]["protocol_train"] = out_rel
with open(CFG, "w") as f:
    yaml.safe_dump(cfg_now, f, sort_keys=False)
print(f"\n[write] {out_abs}")
print(f"[write] {CFG}  (protocol_train → {out_rel})")

## 셀 11 - Smoke test (200 utterances × 2 epoch)
wiring / loss / gradient 확인용. loss가 줄고 dev EER < 40%면 통과.

In [ ]:
# smoke test = wiring 검증 (--no-wandb). 현재 COMBO 구조로 배선/loss/gradient 확인.
# 로컬 ckpt/smoke_<COMBO>/train_log.jsonl · run_meta.json 생성.
!python train.py \
    --config $CFG_REL \
    --tag smoke_$COMBO \
    --adapter $ADAPTER --classifier $CLASSIFIER \
    --smoke 200 \
    --epochs 2 \
    --no-wandb

## 셀 12 - Sanity probe (선택)
adapter와 worker를 freeze하고 SSL layer weight + classifier만 300 step 학습한다.
dev EER 20–30% 정도가 정상이다. 시간이 없으면 셀 13으로 넘어간다.

In [ ]:
import sys, yaml, numpy as np, torch
from sklearn.metrics import roc_curve
# '실험 선택' 셀을 먼저 실행 (CFG 사용).

sys.path.insert(0, PROJECT_ROOT)
from data.dataset import build_dataloaders
from models.model import SSLMultiTaskModel

with open(CFG) as f:                       # 현재 조합의 config
    cfg = yaml.safe_load(f)
cfg["train"]["batch_size"] = min(cfg["train"]["batch_size"], 8)

device = torch.device("cuda")
loaders = build_dataloaders(cfg)
model = SSLMultiTaskModel(cfg).to(device)

for p in model.adapter.parameters(): p.requires_grad = False
for p in model.workers.parameters(): p.requires_grad = False
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"sanity probe trainable params: {sum(p.numel() for p in trainable)/1e3:.1f}K")

opt = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
bce = torch.nn.BCEWithLogitsLoss()

# ~300 step
model.train()
for i, batch in enumerate(loaders["train"]):
    if i >= 300:
        break
    wav = batch["distorted"].to(device, non_blocking=True)
    lab = batch["label"].to(device).float()
    logit, _, _ = model(wav, apply_specaug=False)
    loss = bce(logit, lab)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if i % 50 == 0:
        print(f"step {i:3d}  loss {loss.item():.4f}")

model.eval()
scores, labels = [], []
with torch.no_grad():
    # dev에서 처음 5000 utterances만 빠르게 평가
    seen = 0
    for batch in loaders["dev"]:
        wav = batch["distorted"].to(device)
        logit, _, _ = model(wav, apply_specaug=False)
        scores.extend(torch.sigmoid(logit).cpu().numpy())
        labels.extend(batch["label"].numpy())
        seen += wav.shape[0]
        if seen >= 5000:
            break
scores = np.array(scores); labels = np.array(labels)
fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
fnr = 1.0 - tpr
idx = int(np.nanargmin(np.abs(fnr - fpr)))
eer = (fnr[idx] + fpr[idx]) / 2
print(f"\n[Sanity] dev (first 5K) EER: {eer*100:.2f}%  (expected 20–30%)")
if eer > 0.4:
    print("EER이 40%를 넘었습니다. protocol 파싱·label 인코딩 점검 필요.")

del model, opt, loaders
torch.cuda.empty_cache()

## 셀 12.5 - 학습 설정 출력
학습 직전에 이번 run의 주요 설정과 나머지 설정을 출력한다. train.py도 같은 값을 `config_used.yaml`로 저장한다.

In [ ]:
import yaml, os
# '실험 선택' + 'config 빌드'(+subset) 셀을 먼저 실행 (CFG 파일이 있어야 함).
with open(CFG) as f:
    _cfg = yaml.safe_load(f)

def _flat(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(_flat(v, key + "."))
        else:
            out[key] = v
    return out
flat = _flat(_cfg)

# 주요 학습 설정 (실험 비교의 핵심 축) : (표시이름, 값, 원본 dotted-key)
KEY = [
    ("COMBO",               COMBO,                                    None),
    ("adapter",             flat.get("model.adapter.type"),          "model.adapter.type"),
    ("classifier",          flat.get("model.classifier.type"),       "model.classifier.type"),
    ("adapter.hidden_dim",  flat.get("model.adapter.hidden_dim"),    "model.adapter.hidden_dim"),
    ("ssl",                 flat.get("model.ssl.name_or_path"),      "model.ssl.name_or_path"),
    ("epochs",              flat.get("train.epochs"),                "train.epochs"),
    ("batch_size",          flat.get("train.batch_size"),            "train.batch_size"),
    ("lr_peak",             flat.get("train.lr_peak"),               "train.lr_peak"),
    ("warmup_ratio",        flat.get("train.warmup_ratio"),          "train.warmup_ratio"),
    ("weight_decay",        flat.get("train.weight_decay"),          "train.weight_decay"),
    ("loss.lambda_workers", flat.get("loss.lambda_workers"),         "loss.lambda_workers"),
    ("label_smoothing",     flat.get("model.classifier.label_smoothing"), "model.classifier.label_smoothing"),
    ("train subset",        flat.get("data.protocol_train"),         "data.protocol_train"),
    ("peak_normalize",      flat.get("data.peak_normalize"),         "data.peak_normalize"),
    ("augment.rawboost_p",  flat.get("augment.rawboost_p"),          "augment.rawboost_p"),
    ("augment.specaug_p",   flat.get("augment.specaug_p"),           "augment.specaug_p"),
    ("augment.musan_p",     flat.get("augment.musan_p"),             "augment.musan_p"),
    ("augment.rir_p",       flat.get("augment.rir_p"),               "augment.rir_p"),
    ("augment.codec_p",     flat.get("augment.codec_p"),             "augment.codec_p"),
    ("minDCF cost (miss,fa,pi_spf)",
     f"{flat.get('eval.mindcf_c_miss')}, {flat.get('eval.mindcf_c_fa')}, {flat.get('eval.mindcf_p_target')}",
     None),
    ("seed",                flat.get("seed"),                        "seed"),
]

bw = 70
print("═" * bw)
print(f" 학습 설정  ▏ TAG = {TAG}")
print(f"           ▏ config = {CFG_REL}")
print(f"           ▏ 저장 → {CKPT_DIR}/{TAG}   (W&B: asvspoof5-track1/{TAG})")
_resume = os.path.isfile(f"{CKPT_BACKUP}/{TAG}/last.pt") or os.path.isfile(f"{CKPT_DIR}/{TAG}/last.pt")
print(f"           ▏ resume = {'YES - last.pt 존재 → 이어서 학습' if _resume else 'no - 새 학습'}")
print("═" * bw)
print(" 주요 설정 (key settings)")
shown = {"eval.mindcf_c_miss", "eval.mindcf_c_fa", "eval.mindcf_p_target"}
for name, val, dotted in KEY:
    print(f"   {name:<30}= {val}")
    if dotted:
        shown.add(dotted)

print("\n 기타 설정 (other settings)")
for k in sorted(flat):
    if k in shown:
        continue
    print(f"   {k:<34}= {flat[k]}")
print("═" * bw)

## 셀 13 - 본 학습
조합과 epoch은 셀 9.5 값을 쓴다. per-epoch dev는 5K 랜덤 subset, batch 64로 평가하고 `--save-all-epochs`로 epoch별 ckpt도 남긴다.
5분마다 ckpt를 Drive로 rsync한다 (`epoch_*.pt`는 제외). Drive에 `last.pt`가 있으면 `--resume`을 자동으로 붙여 이어서 학습한다.

In [ ]:
import os, threading, time, subprocess
# '실험 선택' 셀을 먼저 실행 (TAG/CFG_REL/ADAPTER/CLASSIFIER/EPOCHS 사용).

src_dir = f"{CKPT_DIR}/{TAG}/"
dst_dir = f"{CKPT_BACKUP}/{TAG}/"
os.makedirs(src_dir, exist_ok=True)
os.makedirs(dst_dir, exist_ok=True)

# Resume 자동 감지
backup_last = f"{CKPT_BACKUP}/{TAG}/last.pt"
local_last  = f"{CKPT_DIR}/{TAG}/last.pt"
resume_flag = ""
if os.path.isfile(backup_last):
    if not os.path.isfile(local_last):
        subprocess.run(["cp", backup_last, local_last], check=True)
        backup_hist = f"{CKPT_BACKUP}/{TAG}/history.json"
        local_hist  = f"{CKPT_DIR}/{TAG}/history.json"
        if os.path.isfile(backup_hist):
            subprocess.run(["cp", backup_hist, local_hist], check=False)
        print("[resume] copied last.pt (+history.json) from Drive backup")
    resume_flag = f"--resume {local_last}"
else:
    print("[resume] no prior last.pt - starting fresh")

# rsync 백그라운드 (5분마다 /content -> Drive, epoch_*.pt는 제외)
_stop = threading.Event()
def rsync_loop():
    while not _stop.is_set():
        for _ in range(60):
            if _stop.is_set():
                return
            time.sleep(5)
        r = subprocess.run(
            ["rsync", "-a", "--delete", "--exclude=epoch_*.pt", src_dir, dst_dir],
            check=False, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"[backup] {time.strftime('%H:%M:%S')}  rsynced → Drive (epoch_*.pt 제외)")
threading.Thread(target=rsync_loop, daemon=True).start()
print("[backup] background rsync thread started (every 5 min, epoch_*.pt 제외)")

# 학습 실행 - 조합/EPOCHS/config 모두 '실험 선택' 셀에서 옴
# --wandb: 셀 6.5에서 로그인했으면 대시보드 실시간 기록, 아니면 로컬 train_log.jsonl fallback.
print(f"[train] COMBO={COMBO}  adapter={ADAPTER}  classifier={CLASSIFIER}  "
      f"epochs={EPOCHS}  TAG={TAG}", flush=True)
try:
    !python -u train.py \
        --config $CFG_REL \
        --tag $TAG \
        --adapter $ADAPTER --classifier $CLASSIFIER \
        --epochs $EPOCHS \
        --dev-batch-size 64 --dev-subset 5000 \
        --wandb --wandb-project asvspoof5-track1 --wandb-run-name $TAG \
        --save-all-epochs $resume_flag
finally:
    _stop.set()
    subprocess.run(["rsync", "-a", "--delete", "--exclude=epoch_*.pt", src_dir, dst_dir],
                   check=False)
    print(f"\n[backup] final rsync → {dst_dir}  (best/last/history/train_log/run_meta)")
    subprocess.run(["ls", "-lah", src_dir])

## 셀 14 - 학습 곡선 / WavLM layer weight
loss, dev EER, dev minDCF, dev Cllr와 학습된 layer weight 분포를 그린다.

In [ ]:
import json, os, shutil, numpy as np, torch, matplotlib.pyplot as plt
# '실험 선택' 셀을 먼저 실행 (TAG 사용). 다른 조합을 보려면 그 셀에서 COMBO만 바꿔 재실행.

# 세션 재시작 후엔 CKPT_DIR이 비어있음 -> CKPT_BACKUP(Drive)에서 복원
local_dir = f"{CKPT_DIR}/{TAG}"
backup_dir = f"{CKPT_BACKUP}/{TAG}"
if not os.path.isfile(f"{local_dir}/best.pt"):
    if os.path.isfile(f"{backup_dir}/best.pt"):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(backup_dir, local_dir, dirs_exist_ok=True)
        print(f"[restore] {backup_dir} → {local_dir}")
    else:
        raise FileNotFoundError(f"best.pt 없음: {local_dir} 와 {backup_dir} 모두 비어있음")

with open(f"{local_dir}/history.json") as f:
    hist = json.load(f)

epochs     = [h["epoch"] for h in hist]
train_loss = [h["train"]["loss"]   for h in hist]
train_bce  = [h["train"]["bce"]    for h in hist]
train_w    = [h["train"]["worker"] for h in hist]
dev_eer    = [h["dev"]["eer"]*100  for h in hist]

dev_mdcf = [h["dev"].get("min_dcf", float("nan")) for h in hist]
dev_cllr = [h["dev"].get("cllr", float("nan"))   for h in hist]

fig, axes = plt.subplots(1, 4, figsize=(22, 4))
fig.suptitle(f"{TAG}", fontsize=11)
axes[0].plot(epochs, train_loss, label="total")
axes[0].plot(epochs, train_bce,  label="bce")
axes[0].plot(epochs, train_w,    label="worker")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("Train losses")

axes[1].plot(epochs, dev_eer, marker="o", color="crimson")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dev EER (%)")
axes[1].grid(alpha=0.3)
axes[1].set_title(f"dev EER  (min = {min(dev_eer):.2f}%)")

axes[2].plot(epochs, dev_mdcf, marker="o", color="darkorange")
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("dev minDCF (primary)")
axes[2].grid(alpha=0.3)
import numpy as _np
_best_mdcf = _np.nanmin(dev_mdcf) if len(dev_mdcf) else float("nan")
axes[2].set_title(f"dev minDCF  (min = {_best_mdcf:.4f})")

axes[3].plot(epochs, dev_cllr, marker="o", color="steelblue")
axes[3].set_xlabel("epoch"); axes[3].set_ylabel("dev Cllr")
axes[3].grid(alpha=0.3)
axes[3].set_title("dev Cllr (calibration)")

plt.tight_layout(); plt.show()

ckpt = torch.load(f"{local_dir}/best.pt", map_location="cpu")
lw = ckpt["model"]["ssl.layer_weights"]
softmaxed = torch.softmax(lw, dim=0).numpy()

plt.figure(figsize=(13, 3))
plt.bar(range(len(softmaxed)), softmaxed, color="steelblue")
plt.xlabel("WavLM layer index (0 = feature embedding, 24 = top)")
plt.ylabel("softmax weight")
plt.title(f"Learned WavLM layer importance - {TAG}")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

top3 = np.argsort(softmaxed)[-3:][::-1].tolist()
best_ep_mdcf = epochs[int(np.nanargmin(dev_mdcf))] if len(dev_mdcf) else None
print(f"Best dev minDCF : {min(dev_mdcf):.4f}  @ epoch {best_ep_mdcf}")
print(f"Best dev EER    : {min(dev_eer):.2f}%  @ epoch {epochs[int(np.argmin(dev_eer))]}")
print(f"Best ckpt       : best_min_dcf={ckpt.get('best_min_dcf', float('nan')):.4f}, "
      f"best_eer={ckpt.get('best_eer', float('nan'))*100:.2f}%, threshold={ckpt['best_threshold']:.4f}")
print(f"Top-3 layers    : {top3}")

## 셀 14.5 - run_meta / leaderboard
- `run_meta.json`: git commit, dirty 여부, 라이브러리 버전, GPU, 실행 시각
- `scripts/collect_runs.py`: `ckpt/` 아래 모든 run을 dev minDCF 순으로 정리하고 `runs_summary.csv` 생성

In [ ]:
import json, os, subprocess
# '실험 선택' 셀을 먼저 실행 (TAG 사용).
local_dir = f"{CKPT_DIR}/{TAG}"

# 1) 이 run의 재현성 메타데이터
meta_path = f"{local_dir}/run_meta.json"
if os.path.isfile(meta_path):
    meta = json.load(open(meta_path))
    g = meta.get("git", {}); e = meta.get("env", {})
    print(f"── run_meta ({TAG}) ──")
    print(f"  commit  : {g.get('short_commit')}  (dirty={g.get('dirty')}, branch={g.get('branch')})")
    print(f"  when    : {meta.get('timestamp')}")
    print(f"  gpu     : {e.get('gpu')}  |  torch {e.get('torch')} / cuda {e.get('cuda')}")
    print(f"  command : {meta.get('command')}")
else:
    print(f"[run_meta] 없음: {meta_path}")

# 2) step 로그가 로컬 JSONL로 남았는지 (W&B 미사용/오프라인이어도 존재)
jsonl = f"{local_dir}/train_log.jsonl"
if os.path.isfile(jsonl):
    n = sum(1 for _ in open(jsonl))
    print(f"\n[train_log] {jsonl}  ({n} lines: step/epoch/summary 로그)")

# 3) 전체 실험 leaderboard (모든 조합을 dev minDCF 정렬) -> runs_summary.csv 도 생성
print("\n── leaderboard (all runs under ckpt/) ──")
os.chdir(PROJECT_ROOT)
subprocess.run(["python", "scripts/collect_runs.py", "--ckpt-root", CKPT_DIR,
                "--out", f"{CKPT_BACKUP}/runs_summary.csv"], check=False)

## 셀 15 - eval 데이터 다운로드 + 추출 (flac_E_*)
eval 세트에는 train/dev에 없는 공격과 코덱 조건이 들어 있다. 셀 7과 같은 방식으로 받는다.
발화가 약 68만 개라 디스크가 큰 런타임에서 돌린다. 부족하면 eval protocol을 잘라서 평가해야 한다.

In [ ]:
# 셀 15 - Eval 스트리밍 다운로드 (flac_E_*, protocols)
import os, subprocess, shutil
from huggingface_hub import HfApi, hf_hub_download

REPO = "jungjee/asvspoof5"

def disk_free_gb(p="/content"):
    _t, _u, f = shutil.disk_usage(p); return f / 1024**3

def find_dir(prefix, root):
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            if d.startswith(prefix):
                return os.path.join(r, d)
    return None

def find_file(name, root):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

if find_dir("flac_E", DATA_ROOT):
    print(f"[data] eval flac 이미 추출됨 → skip  ({find_dir('flac_E', DATA_ROOT)})")
else:
    api = HfApi()
    files = api.list_repo_files(REPO, repo_type="dataset")
    eval_sh = sorted(f for f in files if f.startswith("flac_E") and f.endswith(".tar"))
    proto   = sorted(f for f in files if f.startswith("ASVspoof5_protocols.tar"))
    print(f"[stream] eval shards={len(eval_sh)}  protocols={len(proto)}  free={disk_free_gb():.0f}GB")
    if not eval_sh:
        raise RuntimeError("HF repo 에 flac_E_*.tar 가 없습니다 - api.list_repo_files 결과를 확인하세요.")

    def stream(fname):
        print(f"  {fname:<30} (free {disk_free_gb():5.0f}GB)", flush=True)
        path = hf_hub_download(REPO, fname, repo_type="dataset", local_dir=RAW_DIR)
        flag = "-xzf" if path.endswith(".gz") else "-xf"
        subprocess.run(["tar", flag, path, "-C", DATA_ROOT], check=True)
        os.remove(path)                                                     # 추출 즉시 tar 삭제
        shutil.rmtree(os.path.join(RAW_DIR, ".cache"), ignore_errors=True)  # hf 임시 blob 정리
        print(f"    extracted + tar 삭제, free {disk_free_gb():.0f}GB", flush=True)

    # eval 프로토콜(키 포함)이 아직 없으면 protocols tar 먼저 확보
    if not find_file("ASVspoof5.eval.track_1.tsv", DATA_ROOT) and proto:
        stream(proto[0])
    for f in eval_sh:
        stream(f)
    print("[stream] eval done")

## 셀 16 - eval 평가 (official metric)
`best.pt`를 eval 세트에서 공식 방식(앞 4초 한 구간, `--sliding` 없음)으로 평가한다. 라벨이 공개돼 있으면 EER/minDCF가 나오고, 숨겨져 있으면 score TSV만 만든다.
수 시간이 걸리므로 처음에는 `--limit 2000`으로 확인해 보는 게 좋다.

In [ ]:
# 셀 16 - Eval 평가 실행
# '실험 선택' 셀을 먼저 실행 (TAG/CFG 사용). 조합 바꾸려면 그 셀에서 COMBO만 변경.
import os, yaml, subprocess, shutil
os.chdir(PROJECT_ROOT)

def find_dir(prefix, root):
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            if d.startswith(prefix):
                return os.path.join(r, d)
    return None

def find_file(name, root):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

# 1) eval 오디오 디렉터리 & 프로토콜(키) 위치 파악
eval_dir = find_dir("flac_E", DATA_ROOT)
eval_tsv = find_file("ASVspoof5.eval.track_1.tsv", DATA_ROOT)
if eval_dir is None:
    raise FileNotFoundError("flac_E_* 디렉터리 없음 - 위 다운로드 셀(셀 15)을 먼저 실행하세요.")
if eval_tsv is None:
    raise FileNotFoundError("ASVspoof5.eval.track_1.tsv 없음 - eval 프로토콜(키)이 repo/추출물에 있는지 확인하세요.")

root              = os.path.dirname(os.path.dirname(eval_tsv))
protocol_eval_rel = os.path.relpath(eval_tsv, root)
audio_eval_rel    = os.path.relpath(eval_dir, root)
n_eval = int(subprocess.check_output(["wc", "-l", eval_tsv]).split()[0])
print(f"[eval] TAG={TAG}")
print(f"[eval] root={root}  protocol={protocol_eval_rel} ({n_eval:,} rows)  audio={audio_eval_rel}")

# 2) 이 조합의 config 에 eval 경로 반영
with open(CFG) as f:
    cfg = yaml.safe_load(f)
cfg["data"]["root"]          = root
cfg["data"]["protocol_eval"] = protocol_eval_rel
cfg["data"]["audio_eval"]    = audio_eval_rel
with open(CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

# 3) 체크포인트 복원(세션 재시작 대비)
local_dir, backup_dir = f"{CKPT_DIR}/{TAG}", f"{CKPT_BACKUP}/{TAG}"
if not os.path.isfile(f"{local_dir}/best.pt"):
    if os.path.isfile(f"{backup_dir}/best.pt"):
        os.makedirs(local_dir, exist_ok=True)
        print("[restore] Drive에서 체크포인트 복사 중… (출력 없이 수 분 걸릴 수 있음)", flush=True)
        shutil.copytree(backup_dir, local_dir, dirs_exist_ok=True)
        print(f"[restore] 완료: {backup_dir} → {local_dir}", flush=True)
    else:
        raise FileNotFoundError(f"best.pt 없음: {local_dir}, {backup_dir} 모두 비어있음")
else:
    print("[restore] 로컬 best.pt 존재 → 복사 skip", flush=True)

# 4) 평가 실행 - 공식 ASVspoof5 방식(단일 세그먼트, --sliding 없음) + 배치화.
#    evaluate.py는 체크포인트에 저장된 학습 config로 모델을 만들므로 조합이 자동으로 맞음.
#    처음엔 --limit 2000 으로 소규모 확인 권장.
os.makedirs("scores", exist_ok=True)
CKPT = f"{CKPT_DIR}/{TAG}/best.pt"
OUT  = f"scores/eval_{TAG}.tsv"
print(f"\n[eval] evaluate.py 시작 (공식 방식, 단일 세그먼트)\n", flush=True)
!python -u evaluate.py \
    --config "$CFG" \
    --ckpt "$CKPT" \
    --split eval \
    --metric official \
    --batch-size 64 \
    --out "$OUT"

# 5) 결과 백업
subprocess.run(["rsync", "-a", "scores/", f"{CKPT_BACKUP}/{TAG}/scores/"], check=True)
print(f"\n[backup] eval scores saved → {CKPT_BACKUP}/{TAG}/scores/")

## 셀 17 - 무결성 확인 + 공격/코덱별 EER
1. 점수 파일에서 pooled EER/minDCF를 다시 계산해 summary json(4.18% / 0.1143)과 맞는지 확인한다.
2. 공격별 EER: 각 attack_tag에 대해 bonafide vs 해당 공격으로 EER 계산. pooled 값이 특정 공격의 큰 오류를 가리고 있지 않은지 본다.
3. 코덱별 EER: 코덱 조건 안에서 bonafide vs spoof.

In [ ]:
# 셀 17 - 공격/코덱별 EER 분해
# '실험 선택' 셀을 먼저 실행 (TAG 사용).
import os, json, numpy as np, pandas as pd
os.chdir(PROJECT_ROOT)
from utils.metrics import compute_eer, compute_min_dcf

def _first(paths):
    return next((p for p in paths if os.path.isfile(p)), None)

def find_file(name, root):
    for r, _, fs in os.walk(root):
        if name in fs:
            return os.path.join(r, name)
    return None

# 점수/요약 파일 (로컬 -> Drive 백업 순으로 탐색)
score_path = _first([f"scores/eval_{TAG}.tsv",
                     f"{CKPT_BACKUP}/{TAG}/scores/eval_{TAG}.tsv"])
summ_path  = _first([f"scores/eval_{TAG}_summary.json",
                     f"{CKPT_BACKUP}/{TAG}/scores/eval_{TAG}_summary.json"])
if score_path is None:
    raise FileNotFoundError(f"eval score tsv 못 찾음 ({TAG}) - 셀 16 을 먼저 돌리세요.")
print(f"[scores]  {score_path}")

# 점수 로드: score(클수록 bonafide), label(1=spoof,0=bona,-1=hidden)
sc = pd.read_csv(score_path, sep="\t", header=None, names=["file_id", "score", "label"])
lab = sc["label"].values
m = lab != -1
score_bona = sc["score"].values[m]
y_bona = (lab[m] == 0).astype(int)          # 1 = bonafide

eer_all, _ = compute_eer(score_bona, y_bona)
mdcf_all, _ = compute_min_dcf(score_bona, y_bona, 1.0, 10.0, 0.05)
print(f"[pooled] {TAG}  EER={eer_all*100:.2f}%  minDCF={mdcf_all:.4f}")
if summ_path:
    s = json.load(open(summ_path))
    print(f"[summary] EER={s.get('eer', float('nan'))*100:.2f}%  minDCF={s.get('min_dcf', float('nan')):.4f}  "
          f"(evaluate.py 산출값 - pooled 재계산과 일치해야 함)")

# 프로토콜에서 attack_tag / codec join
proto = find_file("ASVspoof5.eval.track_1.tsv", DATA_ROOT)
if proto is None:
    raise FileNotFoundError("eval 프로토콜(tsv) 못 찾음.")
pr = pd.read_csv(proto, sep=r"\s+", header=None, engine="python")
pr = pr.rename(columns={1: "file_id", 3: "codec", 6: "attack_tag"})[["file_id", "codec", "attack_tag"]]
df = sc.merge(pr, on="file_id", how="left")
print(f"[join] {df['attack_tag'].notna().mean()*100:.1f}% 행에 attack_tag 매칭됨")

bona = df[df.label == 0]

def eer_bona_vs(subset):
    s = np.concatenate([bona.score.values, subset.score.values])
    y = np.concatenate([np.ones(len(bona)), np.zeros(len(subset))])
    return compute_eer(s, y)[0] * 100

# 공격별 EER (bonafide vs 각 공격)
rows = []
for a in sorted(df.loc[df.label == 1, "attack_tag"].dropna().unique()):
    sp = df[(df.label == 1) & (df.attack_tag == a)]
    rows.append((a, len(sp), eer_bona_vs(sp)))
per_attack = (pd.DataFrame(rows, columns=["attack", "n_spoof", "EER_%"])
              .sort_values("EER_%", ascending=False).reset_index(drop=True))
print("\n══ 공격별 EER (bonafide vs single attack) ══")
print(per_attack.to_string(index=False, float_format=lambda x: f"{x:6.2f}"))

# 코덱별 EER (코덱 조건 내 bonafide vs spoof)
crows = []
for cdc in sorted(df["codec"].dropna().unique()):
    sub = df[(df.codec == cdc) & (df.label != -1)]
    nb, ns = int((sub.label == 0).sum()), int((sub.label == 1).sum())
    if nb == 0 or ns == 0:
        continue
    yb = (sub.label.values == 0).astype(int)
    crows.append((cdc, nb, ns, compute_eer(sub.score.values, yb)[0] * 100))
per_codec = (pd.DataFrame(crows, columns=["codec", "n_bona", "n_spoof", "EER_%"])
             .sort_values("EER_%", ascending=False).reset_index(drop=True))
print("\n══ 코덱별 EER (within-codec) ══")
print(per_codec.to_string(index=False, float_format=lambda x: f"{x:6.2f}"))

# 막대그래프
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f"{TAG}", fontsize=11)
ax[0].barh(per_attack["attack"], per_attack["EER_%"], color="crimson")
ax[0].axvline(eer_all * 100, color="k", ls="--", lw=1, label=f"pooled {eer_all*100:.2f}%")
ax[0].set_xlabel("EER (%)"); ax[0].set_title("공격별 EER"); ax[0].legend(); ax[0].invert_yaxis()
ax[1].barh(per_codec["codec"].astype(str), per_codec["EER_%"], color="darkorange")
ax[1].axvline(eer_all * 100, color="k", ls="--", lw=1)
ax[1].set_xlabel("EER (%)"); ax[1].set_title("코덱별 EER"); ax[1].invert_yaxis()
plt.tight_layout()
os.makedirs("scores", exist_ok=True)
fig.savefig(f"scores/eval_{TAG}_breakdown.png", dpi=120, bbox_inches="tight")
plt.show()

per_attack.to_csv(f"scores/eval_{TAG}_per_attack.csv", index=False)
per_codec.to_csv(f"scores/eval_{TAG}_per_codec.csv", index=False)
import subprocess
subprocess.run(["rsync", "-a", "scores/", f"{CKPT_BACKUP}/{TAG}/scores/"], check=False)
print(f"\n[saved] scores/eval_{TAG}_per_attack.csv , _per_codec.csv , _breakdown.png")

## 셀 18 - actDCF
저장된 eval 점수로 actDCF를 계산한다 (재추론 없음). actDCF는 고정 임계값 τ_Bayes = −ln(β) (β≈1.90)에서의 비용이다.
raw score(−logit)는 LLR로 보정돼 있지 않아 actDCF가 minDCF보다 훨씬 크게 나온다. Eval plan §4.3에서 dev로 calibration하는 것을 허용하므로, dev 점수가 있으면 로지스틱 회귀로 보정한 actDCF도 같이 출력한다 (minDCF/EER은 변하지 않음).

In [ ]:
# 셀 18 - actDCF (raw + dev-calibrated). eval 재추론 없이 저장된 점수 tsv 사용
import os, numpy as np, pandas as pd
os.chdir(PROJECT_ROOT)
from utils.metrics import compute_actdcf, compute_min_dcf, compute_eer

TAG = "a1c1_v2_subset50"
C = dict(c_miss=1.0, c_fa=10.0, p_spoof=0.05)

def _first(ps):
    return next((p for p in ps if os.path.isfile(p)), None)

def _load(path):
    df = pd.read_csv(path, sep="\t", header=None, names=["file_id", "score", "label"])
    m = df.label.values != -1
    s = df.score.values[m]                      # 클수록 bonafide (= -logit)
    y = (df.label.values[m] == 0).astype(int)   # 1 = bonafide
    return s, y

# eval 점수 로드
ev = _first([f"scores/eval_{TAG}.tsv", f"{CKPT_BACKUP}/{TAG}/scores/eval_{TAG}.tsv"])
if ev is None:
    raise FileNotFoundError("eval score tsv 없음 - 셀 16 을 먼저 돌리세요.")
s, y = _load(ev)
print(f"[eval] {ev}  (bonafide={int(y.sum())}, spoof={int((y==0).sum())})")

eer, _        = compute_eer(s, y)
mdcf, _       = compute_min_dcf(s, y, **C)
adcf, tau     = compute_actdcf(s, y, **C)
print(f"\n[raw scores]  EER={eer*100:.2f}%  minDCF={mdcf:.4f}  "
      f"actDCF={adcf:.4f}  (τ_Bayes={tau:.4f})")
print(f"              calibration gap (actDCF − minDCF) = {adcf - mdcf:.4f}  "
      f"← 클수록 점수가 LLR 로 안 맞춰졌다는 뜻")

# (선택) dev 로 선형 캘리브레이션 후 actDCF
dv = _first([f"scores/dev_{TAG}.tsv", f"scores/dev_.tsv",
             f"{CKPT_BACKUP}/{TAG}/scores/dev_{TAG}.tsv",
             f"{CKPT_BACKUP}/{TAG}/scores/dev_.tsv"])
if dv:
    ds, dy = _load(dv)
    from sklearn.linear_model import LogisticRegression
    lr = LogisticRegression(C=1e6, solver="lbfgs")   # Platt/linear calibration
    lr.fit(ds.reshape(-1, 1), dy)
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])   # calibrated LLR = a·score + b
    adcf_c, _ = compute_actdcf(a * s + b, y, **C)
    print(f"\n[dev-calibrated]  actDCF={adcf_c:.4f}   (LLR = {a:.3f}·score + {b:.3f})")
    print(f"                  → minDCF/EER 은 {mdcf:.4f} / {eer*100:.2f}% 로 불변 (보정은 순위 무관)")
    print(f"                  dev calibration 파일: {dv}")
else:
    print("\n[calib] dev 점수 tsv 없음 → raw actDCF 만 계산. "
          "dev eval(셀 15 dev 버전 or evaluate.py --split dev)을 돌리면 캘리브레이션 가능.")

# 요약 저장 + 백업
import json, subprocess
os.makedirs("scores", exist_ok=True)
summ = {"EER_pct": eer * 100, "minDCF": mdcf, "actDCF_raw": adcf, "tau_bayes": tau}
if dv:
    summ["actDCF_dev_calibrated"] = adcf_c
json.dump(summ, open(f"scores/eval_{TAG}_actdcf.json", "w"), indent=2)
subprocess.run(["rsync", "-a", "scores/", f"{CKPT_BACKUP}/{TAG}/scores/"], check=False)
print(f"\n[saved] scores/eval_{TAG}_actdcf.json")

## 참고

기대치 (실측 아님)

| 지표 | 기대값 |
|---|---|
| dev EER | 5–10% |
| dev minDCF | < 0.3 |

학습 곡선 볼 때
- `train.bce`는 감소, `train.worker`는 천천히 감소
- dev minDCF가 일찍 plateau면 early stopping이 걸린다
- WavLM layer weight가 중간층(8–16)에 몰리는지 확인

다른 조합은 셀 9.5에서 `COMBO`만 바꾸고 그 아래 셀을 다시 실행한다.

세션이 끊기면 셀 1–10.5를 다시 실행한 뒤 셀 13을 돌리면 Drive의 `last.pt`에서 이어진다. 평가만 할 때는 셀 9.5를 실행하고 셀 14 이후만 돌린다 (ckpt는 `CKPT_BACKUP`에서 자동 복원).